<a href="https://colab.research.google.com/github/Denise40/Big-Data/blob/main/Big_Data_RU5340365.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Upload do arquivo `imdb-reviews-pt-br.csv` para dentro do Google Colab

In [1]:
!wget https://raw.githubusercontent.com/N-CPUninter/Big_Data/main/data/imdb-reviews-pt-br.zip -O imdb-reviews-pt-br.zip
!unzip imdb-reviews-pt-br.zip
!rm imdb-reviews-pt-br.zip

--2026-11-25 11:37:27--  https://raw.githubusercontent.com/N-CPUninter/Big_Data/main/data/imdb-reviews-pt-br.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 49549692 (47M) [application/zip]
Saving to: ‘imdb-reviews-pt-br.zip’

imdb-reviews-pt-br. 100%[===================>]  47.25M   296MB/s    in 0.2s    

2026-11-25 11:37:27 (296 MB/s) - ‘imdb-reviews-pt-br.zip’ saved [49549692/49549692]

Archive:  imdb-reviews-pt-br.zip
replace imdb-reviews-pt-br.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: imdb-reviews-pt-br.csv  


## Instalação manual das dependências para uso do pyspark no Google Colab

In [4]:
!apt-get update -y
!apt-get install -y openjdk-17-jdk-headless
!pip install pyspark

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [82.5 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:9 http://archive.ubuntu.com/ubuntu noble-updates/restricted amd64 Packages [2,403 kB]
Get:10 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,843 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/multiverse amd64 Packages [61.7 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,184 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,1

## Importar, instanciar e criar a SparkSession

In [5]:
import os
import setuptools

# Garante que o apt-get está atualizado e instala o OpenJDK 17 se não estiver instalado
if not os.path.exists("/usr/lib/jvm/java-17-openjdk-amd64"):
    print("Atualizando repositórios e instalando o OpenJDK 17...")
    !apt-get update -y > /dev/null
    !apt-get install -y openjdk-17-jdk-headless > /dev/null
    print("Java instalado com sucesso!")

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] += os.pathsep + os.path.join(os.environ["JAVA_HOME"], "bin")

from pyspark.sql import SparkSession

appName = "PySpark Trabalho de Big Data"
master = "local"

spark = SparkSession.builder.appName(appName).master(master).getOrCreate()
print("SparkSession iniciada com sucesso!")

SparkSession iniciada com sucesso!


## Criar spark dataframe do CSV utilizando o método read.csv do spark

In [6]:
imdb_df = spark.read.csv('imdb-reviews-pt-br.csv',
                         header=True,
                         quote="\"",
                         escape="\"",
                         encoding="UTF-8")

### Identificação Pessoal e Filtro Inicial

In [7]:
# Número do RU (obrigatório conforme o roteiro)
contador_SEU_RU = 0
print("Identificação Pessoal (RU):", "5340365")

# Mantém apenas registros válidos com sentimento 'neg' ou 'pos'
df_limpo = imdb_df.filter(imdb_df.sentiment.isin("neg", "pos"))

Identificação Pessoal (RU): 5340365


### Questão 1: Soma dos IDs por Sentimento (MapReduce com RDD)

In [13]:
def map1(x):
  # Retorna uma tupla (sentiment, id) acessando como dicionário e convertendo id para int
  return (x["sentiment"], int(x["id"]))


def reduceByKey1(x, y):
  return x + y

# Linha de código para aplicar o map/reduce no seu dataframe spark
resultado = df_limpo.rdd.map(map1).reduceByKey(reduceByKey1).collect()

# Imprimir o resultado
ru_SEU_RU = "5340365"
print(f"RU: {ru_SEU_RU} | Resultado Questão 1: {resultado}")

RU: 5340365 | Resultado Questão 1: [('neg', 459568555), ('pos', 763600041)]


#Questão 2:
###Nesta questão, você irá calcular a diferença no número total de palavras entre textos negativos em português e inglês.

#Objetivo:
###Contar as palavras em cada idioma (colunas 'text_pt' e 'text_en') para entradas onde o sentimento ('sentiment') é 'neg'.
###Subtrair o total de palavras em inglês do total em português.

In [17]:
def map2(x):
  qtd_en = len(str(x["text_en"]).split()) if x["text_en"] is not None else 0
  qtd_pt = len(str(x["text_pt"]).split()) if x["text_pt"] is not None else 0
  return (x["sentiment"], (qtd_en, qtd_pt))

#Cria funções de REDUCE:
def reduceByKey2(x, y):
  # x e y são tuplas (qtd_palavras_en, qtd_palavras_pt) vindas de map2
  return (x[0] + y[0], x[1] + y[1])

# Aplicação do map/reduce e visualização do resultado
resultado = imdb_df.rdd.map(map2).reduceByKey(reduceByKey2).collect()

#imprimir o resultado.
print(f"RU: {ru_SEU_RU} | Resultado bruto Questão 2: {resultado}")

for sentimento, totais in resultado:
  if sentimento == "neg":
    total_en = totais[0]
    total_pt = totais[1]
    diferenca_pt_menos_en = total_pt - total_en
    print(f"Sentimento: {sentimento}")
    print(f"Total de palavras em inglês (text_en): {total_en}")
    print(f"Total de palavras em português (text_pt): {total_pt}")
    print(
        "Diferença (Português - Inglês): "
        f"{total_pt} - {total_en} = {diferenca_pt_menos_en}"
    )

RU: 5340365 | Resultado bruto Questão 2: [('neg', (5400324, 5455273)), ('pos', (5414747, 5462204))]
Sentimento: neg
Total de palavras em inglês (text_en): 5400324
Total de palavras em português (text_pt): 5455273
Diferença (Português - Inglês): 5455273 - 5400324 = 54949
